# 🦈 Pandas Assignment — Shark Tank India Investment Analysis
### Module 01 | neural-nomad-ai-lab

**Scenario:** Analysing all 117 pitches from Shark Tank India Season 1 to understand what makes a successful pitch, how sharks invest, and what patterns exist in the data.

**Dataset columns:**
- `brand_name`, `idea`, `episode_number`, `pitch_number`
- `deal` — 1 if deal made, 0 if not
- `pitcher_ask_amount`, `ask_equity`, `ask_valuation`
- `deal_amount`, `deal_equity`, `deal_valuation`

In [1]:
import pandas as pd
import numpy as np

base = "https://raw.githubusercontent.com/neuralnomad26/neural-nomad-ai-lab/main/modules/01-numpy-pandas/datasets/"

pitches = pd.read_csv(base + "shark_tank_india_pitches.csv")
sharks  = pd.read_csv(base + "shark_tank_india_sharks.csv")

print("Pitches:", pitches.shape)
print("Sharks: ", sharks.shape)

Pitches: (117, 11)
Sharks:  (117, 18)


## Section 1 — Data Inspection
Always understand the structure before doing anything else.

In [2]:
# Task 1.1 — structure overview
pitches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   episode_number      117 non-null    int64  
 1   pitch_number        117 non-null    int64  
 2   brand_name          117 non-null    object 
 3   idea                117 non-null    object 
 4   deal                117 non-null    int64  
 5   pitcher_ask_amount  117 non-null    float64
 6   ask_equity          117 non-null    float64
 7   ask_valuation       117 non-null    float64
 8   deal_amount         117 non-null    float64
 9   deal_equity         117 non-null    float64
 10  deal_valuation      117 non-null    float64
dtypes: float64(6), int64(3), object(2)
memory usage: 10.1+ KB


In [3]:
# Task 1.2 — summary stats for numeric columns
print(pitches.describe().round(2).to_string())

       episode_number  pitch_number    deal  pitcher_ask_amount  ask_equity  ask_valuation  deal_amount  deal_equity  deal_valuation
count          117.00        117.00  117.00              117.00      117.00         117.00       117.00       117.00          117.00
mean            18.74         59.00    0.56              319.85        5.19        3852.46        31.98         8.96          467.10
std             10.07         33.92    0.50             2767.84        3.89       11931.60        36.69        13.11          919.99
min              1.00          1.00    0.00                0.00        0.25           0.01         0.00         0.00            0.00
25%             10.00         30.00    0.00               45.00        2.50         666.67         0.00         0.00            0.00
50%             19.00         59.00    1.00               50.00        5.00        1250.00        25.00         3.00          100.00
75%             27.00         88.00    1.00               80.00      

In [4]:
# Task 1.3 — missing value audit
missing = pd.DataFrame({
    'missing_count': pitches.isnull().sum(),
    'missing_pct':   (pitches.isnull().sum() / len(pitches) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print("Missing value audit:")
print(missing)
print("\nGreat — no missing values in this dataset!")

Missing value audit:
                    missing_count  missing_pct
episode_number                  0          0.0
pitch_number                    0          0.0
brand_name                      0          0.0
idea                            0          0.0
deal                            0          0.0
pitcher_ask_amount              0          0.0
ask_equity                      0          0.0
ask_valuation                   0          0.0
deal_amount                     0          0.0
deal_equity                     0          0.0
deal_valuation                  0          0.0

Great — no missing values in this dataset!


In [5]:
# Task 1.4 — unique values | Task 1.5 — duplicates
print("Unique values per column:")
print(pitches.nunique().sort_values(ascending=False))

print("\nDuplicate rows:", pitches.duplicated().sum())

Unique values per column:
pitch_number          117
brand_name            117
idea                  117
ask_valuation          48
deal_valuation         39
episode_number         35
deal_equity            28
pitcher_ask_amount     26
deal_amount            21
ask_equity             17
deal                    2
dtype: int64

Duplicate rows: 0


## Section 2 — Indexing, Slicing & Selection

In [6]:
# Task 2.1 — iloc examples
print("Rows 10-20:")
print(pitches.iloc[10:21][['brand_name', 'deal', 'pitcher_ask_amount']].to_string())

print("\nLast 5 rows, first 4 columns:")
print(pitches.iloc[-5:, :4].to_string())

Rows 10-20:
             brand_name  deal  pitcher_ask_amount
10         JhaJi Achaar     0                50.0
11               Bummer     1                75.0
12          Revamp Moto     1               100.0
13         Hungry Heads     0                50.0
14   Shrawani Engineers     0                10.0
15          Skippi Pops     1                45.0
16         Menstrupedia     1                50.0
17              Hecolll     0               100.0
18   Raising Superstars     0               100.0
19             Torch-it     1                75.0
20        La Kheer Deli     0                50.0

Last 5 rows, first 4 columns:
     episode_number  pitch_number              brand_name                          idea
112              34           113            Wakao Foods                  Jackfruit Meat
113              34           114  Meatyour Protein Bites                 Protein Bites
114              35           115             Truemeds.in  Online Pharmacy (Generics)
115   

In [7]:
# Task 2.2 — all deals made using loc
deals = pitches.loc[
    pitches['deal'] == 1,
    ['brand_name', 'pitcher_ask_amount', 'deal_amount', 'ask_equity', 'deal_equity']
]
print(f"Total deals made: {len(deals)} out of {len(pitches)} pitches")
print()
print(deals.head().to_string())

Total deals made: 65 out of 117 pitches

            brand_name  pitcher_ask_amount  deal_amount  ask_equity  deal_equity
0  BluePine Industries                50.0         75.0         5.0        16.00
1        Booz scooters                40.0         40.0        15.0        50.00
2  Heart up my Sleeves                25.0         25.0        10.0        30.00
3           Tagz Foods                70.0         70.0         1.0         2.75
7            Peeschute                75.0         75.0         4.0         6.00


In [8]:
# Task 2.4 — startups that accepted less than they asked
lower_deals = pitches.loc[
    (pitches['deal'] == 1) & (pitches['deal_amount'] < pitches['pitcher_ask_amount'])
].copy()
lower_deals['deal_gap'] = lower_deals['pitcher_ask_amount'] - lower_deals['deal_amount']

print(f"Startups that accepted a lower deal than they asked: {len(lower_deals)}")
print()
print(lower_deals[['brand_name', 'pitcher_ask_amount', 'deal_amount', 'deal_gap']].head(4).to_string())

Startups that accepted a lower deal than they asked: 11

              brand_name  pitcher_ask_amount  deal_amount  deal_gap
0    BluePine Industries                50.0         75.0     -25.0
9              Hammer Up               100.0         50.0      50.0
22             Skate India              75.0         50.0      25.0
31             Clensta                  50.0         25.0      25.0


## Section 3 — Querying & Filtering

In [9]:
# Task 3.1 — query examples
eq_range = pitches.query("5 <= ask_equity <= 15")
print(f"Equity between 5% and 15%: {len(eq_range)} pitches")

print()
big_no_deal = pitches.query("pitcher_ask_amount > 100 and deal == 0")
print(f"Big asks (>100L) with no deal: {len(big_no_deal)}")
print(big_no_deal[['brand_name', 'pitcher_ask_amount']].head().to_string())

Equity between 5% and 15%: 60 pitches

Big asks (>100L) with no deal: 6
          brand_name  pitcher_ask_amount
30        Gopal's 56             30000.0
59         KetoIndia               125.0
60        Magic lock               120.0
80            Alpino               150.0
96  Shades of Spring               300.0


In [10]:
# Task 3.2 — string filtering with .str.contains()
india_brands = pitches[pitches['brand_name'].str.contains('India', case=False, na=False)]
print("Brands with 'India' in the name:")
print(india_brands['brand_name'].values)

Brands with 'India' in the name:
['Skate India' 'Shark Tank India' 'Bamboo India']


In [11]:
# Task 3.4 — .between() for range filtering
mid_range = pitches[pitches['pitcher_ask_amount'].between(50, 100)]
print("Ask amount range using .between():")
print(f"  50L to 100L: {len(mid_range)} pitches")
print(f"  Avg ask in this range: {mid_range['pitcher_ask_amount'].mean():.2f}L")

Ask amount range using .between():
  50L to 100L: 67 pitches
  Avg ask in this range: 63.66L


## Section 4 — Feature Engineering & Operations
Creating new columns to add analytical value.

In [12]:
# Task 4.1 — create new analytical columns
pitches['deal_gap']     = pitches['pitcher_ask_amount'] - pitches['deal_amount']
pitches['equity_value'] = (pitches['deal_amount'] / (pitches['deal_equity'] / 100)).replace([np.inf, -np.inf], np.nan)
pitches['got_full_ask'] = pitches['pitcher_ask_amount'] == pitches['deal_amount']

print("New columns created: deal_gap, equity_value, got_full_ask")
print()
print(pitches[['brand_name', 'deal_gap', 'got_full_ask']].head().to_string())

New columns created: deal_gap, equity_value, got_full_ask

  brand_name              deal_gap  got_full_ask
0  BluePine Industries     -25.0         False
1  Booz scooters             0.0          True
2  Heart up my Sleeves       0.0          True
3  Tagz Foods                0.0          True
4  OZiva                    25.0         False


In [13]:
# Task 4.2 — categorise pitch size using .apply()
def pitch_grade(amount):
    if amount < 50:    return 'Small'
    elif amount < 100: return 'Medium'
    elif amount < 500: return 'Large'
    else:              return 'Mega'

pitches['pitch_grade'] = pitches['pitcher_ask_amount'].apply(pitch_grade)

print("Pitch grade distribution:")
print(pitches['pitch_grade'].value_counts())
print("\nMost pitches are Medium (50L-100L) — that's the sweet spot for Shark Tank India")

Pitch grade distribution:
pitch_grade
Medium    61
Small     33
Large     22
Mega       1
Name: count, dtype: int64

Most pitches are Medium (50L-100L) — that's the sweet spot for Shark Tank India


In [14]:
# Task 4.4 — normalise ask amount (vectorized, no loop)
ask_min = pitches['pitcher_ask_amount'].min()
ask_max = pitches['pitcher_ask_amount'].max()

pitches['ask_normalised'] = (pitches['pitcher_ask_amount'] - ask_min) / (ask_max - ask_min)

print(f"Normalised ask amount — min: {pitches['ask_normalised'].min()}, max: {pitches['ask_normalised'].max()}")
print(f"Mean normalised value: {pitches['ask_normalised'].mean():.4f}  (skewed right — Gopal's 56 outlier at 30,000L)")

Normalised ask amount — min: 0.0, max: 1.0
Mean normalised value: 0.0107  (skewed right — Gopal's 56 outlier at 30,000L)


## Section 5 — Groupby & Aggregation
This is where real insights come from.

In [15]:
# Task 5.1 — episode-level summary
ep_summary = pitches.groupby('episode_number').agg(
    total_pitches = ('deal', 'count'),
    deals_made    = ('deal', 'sum'),
    avg_ask       = ('pitcher_ask_amount', 'mean'),
).assign(
    success_rate = lambda x: (x['deals_made'] / x['total_pitches'] * 100).round(1),
    avg_ask      = lambda x: x['avg_ask'].round(2)
).sort_values('success_rate', ascending=False)

print("Episodes with 100% deal rate:")
print(ep_summary[ep_summary['success_rate'] == 100].head().to_string())
print(f"\nOverall deal success rate: {pitches['deal'].mean()*100:.1f}%")

Episodes with 100% deal rate:
                total_pitches  deals_made  avg_ask  success_rate
episode_number                                                  
1                           3           3    38.33         100.0
8                           3           3    45.33         100.0
10                          3           3    28.33         100.0
13                          3           3    43.33         100.0
21                          3           3    60.67         100.0

Overall deal success rate: 55.6%


In [16]:
# Task 5.3 — .agg() with multiple functions in one call
multi_agg = pitches.groupby('pitch_grade')['pitcher_ask_amount'].agg(['min', 'max', 'mean', 'count'])
print("Multi-aggregation on ask amount by pitch grade:")
print(multi_agg.to_string())

Multi-aggregation on ask amount by pitch grade:
                  min      max          mean  count
pitch_grade                                        
Large        100.0000    300.0    120.227273     22
Medium        50.0000     90.0     61.081967     61
Mega       30000.0000  30000.0  30000.000000      1
Small          0.0010     47.0     31.878818     33


In [17]:
# Task 5.4 — .transform() to add group-level stats to each row
pitches['grade_avg_ask']   = pitches.groupby('pitch_grade')['pitcher_ask_amount'].transform('mean')
pitches['above_grade_avg'] = pitches['pitcher_ask_amount'] > pitches['grade_avg_ask']

print(f"Pitches above their grade average: {pitches['above_grade_avg'].sum()} out of {len(pitches)}")
print()
print("Sample — brand vs its grade average:")
print(pitches[['brand_name', 'pitch_grade', 'pitcher_ask_amount', 'grade_avg_ask', 'above_grade_avg']].head().to_string())

Pitches above their grade average: 46 out of 117

Sample — brand vs its grade average:
            brand_name pitch_grade  pitcher_ask_amount  grade_avg_ask  above_grade_avg
0  BluePine Industries      Medium                50.0      61.081967            False
3           Tagz Foods      Medium                70.0      61.081967             True
4                OZiva       Large               100.0     120.227273            False
5       Nutraceuticals       Large               100.0     120.227273            False
7            Peeschute      Medium                75.0      61.081967             True


In [18]:
# Task 5.5 — pivot table
pivot = pitches.pivot_table(
    values   = 'deal_amount',
    index    = 'pitch_grade',
    columns  = 'deal',
    aggfunc  = 'mean',
    fill_value = 0
)
print("Pivot — avg deal amount by pitch grade and deal outcome:")
print(pivot.to_string())

Pivot — avg deal amount by pitch grade and deal outcome:
deal                  0          1
pitch_grade                       
Large         0.000000  97.916667
Medium        0.000000  43.217391
Mega          0.000000   0.000000
Small         0.000000  20.736842


## Section 6 — Merging

In [19]:
# Task 6.1 — explore the sharks dataframe
print("Sharks columns:")
print(sharks.columns.tolist())
print()
print("Sample:")
print(sharks[['pitch_number', 'total_sharks_invested', 'amount_per_shark', 'equity_per_shark']].head(3).to_string())

Sharks columns:
['pitch_number', 'ashneer_present', 'anupam_present', 'aman_present', 'namita_present', 'vineeta_present', 'peyush_present', 'ghazal_present', 'ashneer_deal', 'anupam_deal', 'aman_deal', 'namita_deal', 'vineeta_deal', 'peyush_deal', 'ghazal_deal', 'total_sharks_invested', 'amount_per_shark', 'equity_per_shark']

Sample:
   pitch_number  total_sharks_invested  amount_per_shark  equity_per_shark
0             1                      3              25.0          5.333333
1             2                      2              20.0         25.000000
2             3                      2              12.5         15.000000


In [20]:
# Task 6.2 — merge on pitch_number
merged = pitches.merge(sharks, on='pitch_number', how='left')
print(f"Before merge: {pitches.shape}  →  After merge: {merged.shape}")
print("Merge key: pitch_number — every pitch has shark data")

Before merge: (117, 20)  →  After merge: (117, 37)
Merge key: pitch_number — every pitch has shark data


In [21]:
# Task 6.3 — most active shark by number of deals
shark_cols  = ['ashneer_deal','anupam_deal','aman_deal','namita_deal','vineeta_deal','peyush_deal','ghazal_deal']
shark_names = ['Ashneer', 'Anupam', 'Aman', 'Namita', 'Vineeta', 'Peyush', 'Ghazal']

deal_counts = {name: int(merged[col].sum()) for name, col in zip(shark_names, shark_cols)}
sorted_sharks = sorted(deal_counts.items(), key=lambda x: -x[1])

print("Shark deal counts (Season 1):")
for rank, (name, count) in enumerate(sorted_sharks, 1):
    print(f"  {rank}. {name:<8} — {count:2} deals")

Shark deal counts (Season 1):
  1. Aman     — 28 deals
  2. Peyush   — 27 deals
  3. Anupam   — 24 deals
  4. Namita   — 22 deals
  5. Ashneer  — 21 deals
  6. Vineeta  — 15 deals
  7. Ghazal   —  7 deals


In [22]:
# Task 6.4 — split and concat
half       = len(pitches) // 2
first_half = pitches.iloc[:half]
last_half  = pitches.iloc[half:]

rejoined = pd.concat([first_half, last_half]).reset_index(drop=True)
print(f"Original: {pitches.shape} | Rejoined: {rejoined.shape}")
print("✅ Shape confirmed — concat worked correctly")

Original: (117, 20) | Rejoined: (117, 20)
✅ Shape confirmed — concat worked correctly


## Section 7 — Sorting, Ranking & Final Insights

In [23]:
# Task 7.1 — top 10 highest deals
top10 = pitches.sort_values('deal_amount', ascending=False).head(10)
print("Top 10 highest deals:")
print(top10[['brand_name', 'pitcher_ask_amount', 'deal_amount', 'deal_equity']].to_string())

Top 10 highest deals:
               brand_name  pitcher_ask_amount  deal_amount  deal_equity
50          Aas Vidyalaya               150.0        150.0         15.0
36                  Annie                30.0        105.0          3.0
12            Revamp Moto               100.0        100.0          1.5
38        The Yarn Bazaar                50.0        100.0         10.0
39      The Renal Project               100.0        100.0          6.0
42       Hammer Lifestyle                30.0        100.0         40.0
18     Raising Superstars               100.0        100.0          4.0
63               IN A CAN                50.0        100.0         10.0
64             Get a Whey               100.0        100.0         15.0
15            Skippi Pops                45.0        100.0         15.0


In [24]:
# Task 7.2 — rank within episode by ask amount
pitches['episode_rank'] = pitches.groupby('episode_number')['pitcher_ask_amount'] \
    .rank(ascending=False, method='dense')

print("Episode rank sample (rank 1 = highest ask in that episode):")
print(pitches[['brand_name', 'episode_number', 'pitcher_ask_amount', 'episode_rank']].head().to_string())

Episode rank sample (rank 1 = highest ask in that episode):
            brand_name  episode_number  pitcher_ask_amount  episode_rank
0  BluePine Industries               1                50.0           1.0
1        Booz scooters               1                40.0           2.0
2  Heart up my Sleeves               1                25.0           3.0
3           Tagz Foods               2                70.0           2.0
4                OZiva               2               100.0           1.0


In [25]:
# Task 7.3 — deal efficiency (equity % per lakh invested)
deals_only = pitches[pitches['deal'] == 1].copy()
deals_only['deal_efficiency'] = deals_only['deal_equity'] / deals_only['deal_amount'].replace(0, np.nan)

top5_eff = deals_only.nlargest(5, 'deal_efficiency')
print("Top 5 most efficient deals for sharks (highest equity per lakh invested):")
print(top5_eff[['brand_name', 'deal_amount', 'deal_equity', 'deal_efficiency']].to_string())

Top 5 most efficient deals for sharks (highest equity per lakh invested):
               brand_name  deal_amount  deal_equity  deal_efficiency
44                Cocofit      0.00005          5.0     100000.00000
85   Watt Technovations      0.00101          4.0       3960.39604
76            KG Agrotech     10.00000         40.0          4.00000
65          Sid07 Designs     25.00000         75.0          3.00000
1           Booz scooters     40.00000         50.0          1.25000


## 📊 Key Insights from the Analysis

- **55.6% of pitches resulted in a deal** — more than half the startups that appeared walked away with investment.
- **Only 38.5% of founders got their full ask amount** — most deals were negotiated down or restructured with higher equity.
- **Aman Gupta was the most active shark with 28 deals** — nearly involved in half of all successful investments in Season 1.
- **11 startups accepted a lower deal than they asked for** — suggesting sharks pushed back hard on valuations.
- **The biggest deal was Aas Vidyalaya at ₹150L** — they asked for 150L and got exactly what they wanted, rare in the tank.
- **Medium-sized pitches (50L–100L) had the highest volume** — 61 out of 117, making it the most common ask bracket.